In [1]:
import sys
sys.path.append("../")

import pandas as pd
import numpy as np

from src.heston import estimate_heston_parameters

In [2]:
portfolio_returns = pd.read_csv(
    "../data/processed/portfolio_returns.csv",
    index_col=0,
    parse_dates=True
).squeeze()

portfolio_returns.name = "Portfolio Return"

portfolio_returns.head()

Date
2015-01-05   -0.019483
2015-01-06   -0.005603
2015-01-07    0.014929
2015-01-08    0.020986
2015-01-09   -0.008747
Name: Portfolio Return, dtype: float64

In [3]:
heston_params = estimate_heston_parameters(
    portfolio_returns.loc["2015":"2021"]
)

heston_params

{'mu': np.float64(0.1580025038123477),
 'kappa': np.float64(1.6453904729554076),
 'theta': np.float64(0.03423342673834108),
 'xi': np.float64(0.40744330675856055),
 'rho': -0.11407242259432454,
 'v0': np.float64(0.02849351168799935)}

In [4]:
feller_lhs = (
    2
    * heston_params["kappa"]
    * heston_params["theta"]
)

feller_rhs = (
    heston_params["xi"] ** 2
)

print("2κθ =", feller_lhs)
print("ξ²   =", feller_rhs)
print(
    "Feller condition satisfied:",
    feller_lhs > feller_rhs
)

2κθ = 0.11265470842376665
ξ²   = 0.16601004822235046
Feller condition satisfied: False


In [5]:
from src.heston import (
    estimate_heston_parameters,
    simulate_heston
)

In [6]:
simulated_returns_1d = simulate_heston(
    **heston_params,
    horizon=1,
    n_steps=1,
    n_simulations=50000,
    random_state=42
)

In [7]:
print(
    "Mean:",
    simulated_returns_1d.mean()
)

print(
    "Std:",
    simulated_returns_1d.std()
)

print(
    "Minimum:",
    simulated_returns_1d.min()
)

print(
    "Maximum:",
    simulated_returns_1d.max()
)

Mean: 0.0006213538210025539
Std: 0.010661203731735217
Minimum: -0.04604427790179184
Maximum: 0.05387099869746445


In [8]:
print(
    "5% quantile:",
    np.quantile(
        simulated_returns_1d,
        0.05
    )
)

print(
    "1% quantile:",
    np.quantile(
        simulated_returns_1d,
        0.01
    )
)

5% quantile: -0.01691023192872391
1% quantile: -0.024220578185518503


In [9]:
simulated_returns_10d = simulate_heston(
    **heston_params,
    horizon=10,
    n_steps=10,
    n_simulations=50000,
    random_state=42
)

In [10]:
print(
    "10-day mean:",
    simulated_returns_10d.mean()
)

print(
    "10-day volatility:",
    simulated_returns_10d.std()
)

print(
    "10-day 5% quantile:",
    np.quantile(
        simulated_returns_10d,
        0.05
    )
)

print(
    "10-day 1% quantile:",
    np.quantile(
        simulated_returns_10d,
        0.01
    )
)

10-day mean: 0.006214186610763709
10-day volatility: 0.03383789258514256
10-day 5% quantile: -0.05010811359916963
10-day 1% quantile: -0.07543897849691997


In [11]:
train_returns = portfolio_returns.loc[
    "2015":"2021"
]

print(
    "Historical annualized volatility:",
    train_returns.std() * np.sqrt(252)
)

print(
    "Heston 1-day simulated volatility:",
    simulated_returns_1d.std() * np.sqrt(252)
)

Historical annualized volatility: 0.18733943036912223
Heston 1-day simulated volatility: 0.16924136250459093


In [12]:
pd.DataFrame(
    [heston_params]
).to_csv(
    "../output/tables/heston_training_parameters.csv",
    index=False
)

##### **Heston 1-day Validation**

In [13]:
from src.heston import (
    walk_forward_heston,
    calculate_var_es
)

In [14]:
heston_1d_validation = walk_forward_heston(
    portfolio_returns,
    forecast_start="2022-01-03",
    forecast_end="2023-12-29",
    horizon=1,
    n_simulations=50000,
    random_state=42
)

In [15]:
def future_cumulative_return(
    returns,
    date,
    horizon=10
):
    position = returns.index.get_loc(date)

    future_returns = returns.iloc[
        position + 1:
        position + 1 + horizon
    ]

    if len(future_returns) < horizon:
        return np.nan

    return future_returns.sum()

heston_1d_validation["Realized_Return"] = [
    future_cumulative_return(
        portfolio_returns,
        date,
        horizon=1
    )
    for date in heston_1d_validation.index
]

In [16]:
heston_1d_validation = (
    heston_1d_validation
    .dropna()
)

In [17]:
heston_1d_validation["Violation_95"] = (
    heston_1d_validation["Realized_Return"]
    <
    -heston_1d_validation["VaR_95"]
)

heston_1d_validation["Violation_99"] = (
    heston_1d_validation["Realized_Return"]
    <
    -heston_1d_validation["VaR_99"]
)

In [18]:
print(
    "95% violation rate:",
    heston_1d_validation["Violation_95"].mean()
)

print(
    "99% violation rate:",
    heston_1d_validation["Violation_99"].mean()
)

95% violation rate: 0.05389221556886228
99% violation rate: 0.021956087824351298


In [19]:
from src.risk import kupiec_test, christoffersen_conditional_coverage_test

heston_1d_kupiec_95 = kupiec_test(
    heston_1d_validation["Violation_95"],
    confidence_level=0.95
)

heston_1d_kupiec_99 = kupiec_test(
    heston_1d_validation["Violation_99"],
    confidence_level=0.99
)

heston_1d_cc_95 = (
    christoffersen_conditional_coverage_test(
        heston_1d_validation["Violation_95"],
        confidence_level=0.95
    )
)

heston_1d_cc_99 = (
    christoffersen_conditional_coverage_test(
        heston_1d_validation["Violation_99"],
        confidence_level=0.99
    )
)


heston_1d_backtest = pd.DataFrame([
    {
        "Model": "Heston",
        "Confidence": 0.95,
        **heston_1d_kupiec_95,
        **{
            f"CC_{k}": v
            for k, v in heston_1d_cc_95.items()
        }
    },
    {
        "Model": "Heston",
        "Confidence": 0.99,
        **heston_1d_kupiec_99,
        **{
            f"CC_{k}": v
            for k, v in heston_1d_cc_99.items()
        }
    }
])

heston_1d_backtest

,Model,Confidence,n_observations,n_violations,expected_rate,observed_rate,lr_statistic,p_value,CC_lr_uc,CC_lr_ind,CC_lr_cc,CC_p_value
0,Heston,0.95,501,27,0.05,0.053892,0.156004,0.692862,0.156004,0.204149,0.360154,0.835206
1,Heston,0.99,501,11,0.01,0.021956,5.394739,0.020198,5.394739,0.494929,5.889668,0.052611


In [20]:
heston_1d_validation.to_csv(
    "../output/tables/heston_1day_validation_risk.csv"
)

heston_1d_backtest.to_csv(
    "../output/tables/heston_1day_validation_backtest.csv",
    index=False
)

##### **Heston 10-day Validation**

In [21]:
heston_10d_validation = walk_forward_heston(
    portfolio_returns,
    forecast_start="2022-01-03",
    forecast_end="2023-12-15",
    horizon=10,
    n_simulations=50000,
    random_state=42
)

In [22]:
heston_10d_validation["Realized_Return_10D"] = [
    future_cumulative_return(
        portfolio_returns,
        date,
        horizon=10
    )
    for date in heston_10d_validation.index
]

heston_10d_validation = (
    heston_10d_validation
    .dropna()
)

In [23]:
heston_10d_validation["Violation_95"] = (
    heston_10d_validation["Realized_Return_10D"]
    <
    -heston_10d_validation["VaR_95"]
)

heston_10d_validation["Violation_99"] = (
    heston_10d_validation["Realized_Return_10D"]
    <
    -heston_10d_validation["VaR_99"]
)

In [24]:
print(
    "95% violation rate:",
    heston_10d_validation["Violation_95"].mean()
)

print(
    "99% violation rate:",
    heston_10d_validation["Violation_99"].mean()
)

95% violation rate: 0.06504065040650407
99% violation rate: 0.018292682926829267


In [25]:
heston_10d_kupiec_95 = kupiec_test(
    heston_10d_validation["Violation_95"],
    confidence_level=0.95
)

heston_10d_kupiec_99 = kupiec_test(
    heston_10d_validation["Violation_99"],
    confidence_level=0.99
)

heston_10d_cc_95 = (
    christoffersen_conditional_coverage_test(
        heston_10d_validation["Violation_95"],
        confidence_level=0.95
    )
)

heston_10d_cc_99 = (
    christoffersen_conditional_coverage_test(
        heston_10d_validation["Violation_99"],
        confidence_level=0.99
    )
)


heston_10d_backtest = pd.DataFrame([
    {
        "Model": "Heston",
        "Confidence": 0.95,
        **heston_10d_kupiec_95,
        **{
            f"CC_{k}": v
            for k, v in heston_10d_cc_95.items()
        }
    },
    {
        "Model": "Heston",
        "Confidence": 0.99,
        **heston_10d_kupiec_99,
        **{
            f"CC_{k}": v
            for k, v in heston_10d_cc_99.items()
        }
    }
])

heston_10d_backtest

,Model,Confidence,n_observations,n_violations,expected_rate,observed_rate,lr_statistic,p_value,CC_lr_uc,CC_lr_ind,CC_lr_cc,CC_p_value
0,Heston,0.95,492,32,0.05,0.065041,2.149107,0.142653,2.149107,91.629823,93.778930,0.000000
1,Heston,0.99,492,9,0.01,0.018293,2.744761,0.097574,2.744761,21.821642,24.566403,0.000005


##### **Performance on Test Set**

In [26]:
heston_1d_test = walk_forward_heston(
    portfolio_returns,
    forecast_start="2024-01-02",
    forecast_end="2025-12-31",
    horizon=1,
    n_simulations=50000,
    random_state=121
)

In [27]:
heston_10d_test = walk_forward_heston(
    portfolio_returns,
    forecast_start="2024-01-02",
    forecast_end="2025-12-15",
    horizon=10,
    n_simulations=50000,
    random_state=121
)

In [28]:
import pandas as pd

# ==========================================
# 1. PROCESS HESTON 1-DAY TEST SET
# ==========================================

# Add realized returns for 1-day horizon
heston_1d_test["Realized_Return"] = [
    future_cumulative_return(portfolio_returns, date, horizon=1)
    for date in heston_1d_test.index
]
heston_1d_test = heston_1d_test.dropna()

# Compute violations
heston_1d_test["Violation_95"] = (
    heston_1d_test["Realized_Return"] < -heston_1d_test["VaR_95"]
)
heston_1d_test["Violation_99"] = (
    heston_1d_test["Realized_Return"] < -heston_1d_test["VaR_99"]
)

# Run backtests
heston_1d_kupiec_95 = kupiec_test(
    heston_1d_test["Violation_95"], confidence_level=0.95
)
heston_1d_kupiec_99 = kupiec_test(
    heston_1d_test["Violation_99"], confidence_level=0.99
)

heston_1d_cc_95 = christoffersen_conditional_coverage_test(
    heston_1d_test["Violation_95"], confidence_level=0.95
)
heston_1d_cc_99 = christoffersen_conditional_coverage_test(
    heston_1d_test["Violation_99"], confidence_level=0.99
)

# Build summary table for 1-day test backtest
heston_1d_test_backtest = pd.DataFrame([
    {
        "Model": "Heston (1-Day)",
        "Confidence": 0.95,
        **heston_1d_kupiec_95,
        **{f"CC_{k}": v for k, v in heston_1d_cc_95.items()},
    },
    {
        "Model": "Heston (1-Day)",
        "Confidence": 0.99,
        **heston_1d_kupiec_99,
        **{f"CC_{k}": v for k, v in heston_1d_cc_99.items()},
    },
])


# ==========================================
# 2. PROCESS HESTON 10-DAY TEST SET
# ==========================================

# Add realized returns for 10-day horizon
heston_10d_test["Realized_Return_10D"] = [
    future_cumulative_return(portfolio_returns, date, horizon=10)
    for date in heston_10d_test.index
]
heston_10d_test = heston_10d_test.dropna()

# Compute violations
heston_10d_test["Violation_95"] = (
    heston_10d_test["Realized_Return_10D"] < -heston_10d_test["VaR_95"]
)
heston_10d_test["Violation_99"] = (
    heston_10d_test["Realized_Return_10D"] < -heston_10d_test["VaR_99"]
)

# Run backtests
heston_10d_kupiec_95 = kupiec_test(
    heston_10d_test["Violation_95"], confidence_level=0.95
)
heston_10d_kupiec_99 = kupiec_test(
    heston_10d_test["Violation_99"], confidence_level=0.99
)

heston_10d_cc_95 = christoffersen_conditional_coverage_test(
    heston_10d_test["Violation_95"], confidence_level=0.95
)
heston_10d_cc_99 = christoffersen_conditional_coverage_test(
    heston_10d_test["Violation_99"], confidence_level=0.99
)

# Build summary table for 10-day test backtest
heston_10d_test_backtest = pd.DataFrame([
    {
        "Model": "Heston (10-Day)",
        "Confidence": 0.95,
        **heston_10d_kupiec_95,
        **{f"CC_{k}": v for k, v in heston_10d_cc_95.items()},
    },
    {
        "Model": "Heston (10-Day)",
        "Confidence": 0.99,
        **heston_10d_kupiec_99,
        **{f"CC_{k}": v for k, v in heston_10d_cc_99.items()},
    },
])


# ==========================================
# 3. SAVE RESULTS TO OUTPUT TABLES
# ==========================================

heston_1d_test.to_csv("../output/tables/heston_1day_test_risk.csv")
heston_1d_test_backtest.to_csv(
    "../output/tables/heston_1day_test_backtest.csv", index=False
)

heston_10d_test.to_csv("../output/tables/heston_10day_test_risk.csv")
heston_10d_test_backtest.to_csv(
    "../output/tables/heston_10day_test_backtest.csv", index=False
)

# Display backtest summaries
display(heston_1d_test_backtest)
display(heston_10d_test_backtest)

,Model,Confidence,n_observations,n_violations,expected_rate,observed_rate,lr_statistic,p_value,CC_lr_uc,CC_lr_ind,CC_lr_cc,CC_p_value
0,Heston (1-Day),0.95,501,29,0.05,0.057884,0.625349,0.429066,0.625349,0.064315,0.689664,0.708339
1,Heston (1-Day),0.99,501,8,0.01,0.015968,1.526151,0.216691,1.526151,2.569763,4.095915,0.128998


,Model,Confidence,n_observations,n_violations,expected_rate,observed_rate,lr_statistic,p_value,CC_lr_uc,CC_lr_ind,CC_lr_cc,CC_p_value
0,Heston (10-Day),0.95,491,24,0.05,0.04888,0.013063,0.909005,0.013063,62.372545,62.385608,2.842171e-14
1,Heston (10-Day),0.99,491,9,0.01,0.01833,2.761622,0.096550,2.761622,41.880892,44.642514,2.023025e-10


In [29]:
from src.risk import create_backtest_summary

In [30]:
heston_1d_summary = create_backtest_summary(
    heston_1d_test,
    model="Heston",
    horizon=1
)

KeyError: 'CC_lr_cc'